# M3+M4의 중간 순위 손실과 N/V — 새 M2 학습 전 진단
기존 Dunnhumby 개발분할 seed 43·44의 M3와 M3+M4-B, 300 epoch 체크포인트 네 개를 읽습니다. **새 학습은 없으며 CPU 런타임으로 실행할 수 있습니다.**

전체·세그먼트·노출 지표를 원본과 대조한 뒤, 정답의 1–10/11–20/21–50/51–100위 이동, 경계 점수차, N/V·degree 구간별 손실을 저장합니다. 표의 변화는 M3+M4-B − M3입니다.

N/V 구간은 TRAIN 고객에서 정하며 동률을 임의로 나누지 않습니다. 신규상품 과업과 기존 684–690일 개발분할을 그대로 읽습니다. 최종 test·holdout은 만들지 않습니다. 이 결과는 제안한 N/V 방향조절 M2의 성공이나 CLV 인과효과를 판정하지 않으며 자동으로 학습을 시작하지 않습니다.

**모두 실행** 후 마지막 셀에서 ZIP을 내려받아 주세요. 실행시간은 Drive 읽기 속도에 따라 달라집니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
try:
    ROOT.stat()
except OSError:
    drive.mount('/content/drive', force_remount=True)
if not ROOT.is_dir():
    raise RuntimeError(f'데이터 폴더를 확인하세요: {ROOT}')
SOURCE_COMMIT = 'c960b8ea72099914458b567246d6fbbed06e5d33'
REPO = Path('/content/clv-m5-nv-rank-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'clv_m5_nv_rank_diagnostic'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != REPO.resolve():
        raise RuntimeError('다른 실험 소스가 로드돼 있습니다. 이 런타임을 재시작하세요.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_m5_nv_rank_diagnostic as diagnostic
print('준비 완료: 기존 M3/M5-B, seed43·44, epoch300 · 학습 없음')

In [ ]:
paths = diagnostic.run(root=str(ROOT), seeds=(43, 44))
print(json.dumps(paths, ensure_ascii=False, indent=2))

`truth_lost`: M3에서 적중했지만 M3+M4에서는 컷오프 밖으로 나간 정답 수. `lost_to_next5`: 그중 경계 다음 5개 순위 안에 남은 정답 수. `mean_recall_delta`와 `mean_weighted_hit_delta`는 고객 평균 절대차입니다. 상관계수는 기술적 진단이며 N/V에 따른 개입 효과가 아닙니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
zip_path = Path('/content/clv_m5_nv_rank_diagnostic_seed43_44.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))